# Data Preprocessing & Feature Engineering
## AI-Driven Coronary Disease Detection and Decision Support System
# Process 04 – Patient Recommendation System

Import Libraries

In [27]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from google.colab import drive
drive.mount('/content/drive')


Load Dataset

In [28]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DSGP/Heart_health new.csv")
df.head()


,ID,Name,Age,Gender,Height cm,Weight kg,Blood Pressure mmHg,Cholesterol mg/dL,Glucose mg/dL,Smoker,Exercise hours/week,Heart Attack
0,2,Jane Smith,35,Female,160,65,110/70,180,80,No,2,0
1,4,Sarah Brown,40,Female,165,70,115/75,190,85,No,3,0
2,6,Emily Davis,30,Female,155,60,105/65,170,75,No,1,0
3,8,Amanda Martinez,38,Female,162,68,118/72,195,88,No,2,0
4,10,Laura Garcia,42,Female,168,72,120/78,200,90,Yes,3,0


Handle Missing Values

In [37]:
df.isnull().sum()

# Fill missing exercise hours with mean value
"""df['Exercise hours/week'].fillna(df['Exercise hours/week'].mean(), inplace=True)

# Fill missing categorical values with most frequent value
df['Gender'].fillna(df['Gender'].mode()[0], inplace=True)
df['Smoker'].fillna(df['Smoker'].mode()[0], inplace=True)"""


"df['Exercise hours/week'].fillna(df['Exercise hours/week'].mean(), inplace=True)\n\n# Fill missing categorical values with most frequent value\ndf['Gender'].fillna(df['Gender'].mode()[0], inplace=True)\ndf['Smoker'].fillna(df['Smoker'].mode()[0], inplace=True)"

Encode Categorical Variables

In [30]:
# Gender: Male=1, Female=0
df['Gender'] = df['Gender'].map({'Male':1, 'Female':0})

# Smoker: Yes=1, No=0
df['Smoker'] = df['Smoker'].map({'Yes':1, 'No':0})


Split Blood Pressure into Systolic & Diastolic

In [31]:
if 'Blood Pressure mmHg' in df.columns:
    df[['Systolic_BP', 'Diastolic_BP']] = df['Blood Pressure mmHg'] \
        .str.split('/', expand=True).astype(int)
    df.drop('Blood Pressure mmHg', axis=1, inplace=True)
else:
    print("Column 'Blood Pressure mmHg' not found. Assuming it was already processed or is missing.")

Feature Scaling

In [32]:
scaler = MinMaxScaler()
num_cols = ['Age', 'Height cm', 'Weight kg', 'Cholesterol mg/dL',
            'Glucose mg/dL', 'Exercise hours/week', 'Systolic_BP', 'Diastolic_BP']

df[num_cols] = scaler.fit_transform(df[num_cols])


Feature Engineering – BMI

In [33]:
df['BMI'] = df['Weight kg'] / ((df['Height cm']) ** 2)
df['BMI'] = df['BMI'].round(2)


Feature Engineering – Lifestyle Risk Score

In [34]:
# Scale exercise hours (already 0-1 from MinMaxScaler)
df['Lifestyle_Risk'] = (
    df['Smoker']*0.4 +
    (1 - df['Exercise hours/week'])*0.3 +
    df['BMI']*0.3
)
df['Lifestyle_Risk'] = df['Lifestyle_Risk'].round(3)


Feature Engineering – Composite Patient Risk Score

In [35]:
# These columns are assumed to be outputs from previous processes
df['patient_risk_score'] = (
    df.get('Blockage_Percentage', 0) * 0.5 +
    df.get('ECG_Abnormality_Score', 0) * 0.3 +
    df['Age'] * 0.2
)

df['patient_risk_score'] = df['patient_risk_score'].round(3)


In [36]:
selected_features = [
    'Age', 'Gender', 'BMI', 'Lifestyle_Risk',
    'patient_risk_score', 'Systolic_BP', 'Diastolic_BP',
    'Cholesterol mg/dL', 'Glucose mg/dL'
]

X = df[selected_features]
y = df['Heart Attack']   # Used only for historical data